# Hybrid: how much does one attention layer recover?

6-layer models where layer 6 (index 5) is full attention and the other five use a local mixer. The local-only versions are trained alongside, so we can measure **how much of each local mixer's gap to attention one attention layer recovers**. One extra model puts the attention layer in the middle (index 2) instead.

| local mixer | local only | + attention at layer 6 |
|---|---|---|
| `grassmann` (paper) | ✓ | ✓ |
| `concat32` (no geometry, same compression) | ✓ | ✓ (also at layer 3) |
| `hyena` (short filter) | ✓ | ✓ |
| `g-both` (edge-choosing, reach 127) | ✓ | ✓ |
| `attention` (all six layers) | ✓ | |

2 seeds × 5 epochs at LR 1e-3, validation only. About an hour on an A100 (estimate).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, json, subprocess, sys
REPO, DATA, OUTDIR = '/content/grassman', '/content/data', '/content/results'
RUNS = f'{OUTDIR}/hybrid.jsonl'
os.makedirs(OUTDIR, exist_ok=True)
if os.path.exists(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://github.com/thebnbrkr/grassman.git {REPO}
%cd {REPO}
!git log --oneline -1
!pip install -q datasets transformers
# drop any grassman modules imported earlier in this runtime, so the code just pulled is what runs
for _name in [k for k in sys.modules if k == 'grassman' or k.startswith('grassman.')]:
    del sys.modules[_name]

Only if you are **resuming this notebook** in a new runtime: set `RESTORE = True` to upload the `results.zip` it downloaded earlier, so finished runs are skipped. On a first run leave it `False`.

In [ ]:
RESTORE = False
if RESTORE:
    from google.colab import files
    for name in files.upload():
        !cd /content && unzip -o -q "{name}"
!ls -la {OUTDIR}

Unit tests (about a minute). Stop if anything fails.

In [ ]:
!python -m pytest -q

In [ ]:
!python -m grassman.data --data-dir {DATA}

In [ ]:
# Run every model of this notebook on the same GPU type: precision differs (bf16 on A100/L4, fp16 on T4).
# COMPILE = True uses torch.compile for every model (fair across models); turn it on only if speed.ipynb
# showed compiled runs are faster on this GPU, since compiling costs time per run.
COMPILE = False

def train(mixer, seed, epochs=5, lr=1e-3, attn='', save=None, tag='pilot'):
    cmd = [sys.executable, '-m', 'grassman.train', '--mixer', mixer, '--seed', str(seed), '--lr', str(lr),
           '--epochs', str(epochs), '--tag', tag, '--data-dir', DATA, '--out', RUNS,
           '--skip-existing', '--no-test']
    if attn:
        cmd += ['--attn-layers', attn]
    if save:
        cmd += ['--save', save]
    if COMPILE:
        cmd += ['--compile']
    print('>>', mixer, f'attn@{attn}' if attn else '', 'seed', seed, flush=True)
    subprocess.run(cmd, cwd=REPO, check=True)

In [ ]:
CONFIGS = [('attention', ''),
           ('grassmann', ''), ('grassmann', '5'),
           ('concat32', ''), ('concat32', '5'), ('concat32', '2'),
           ('hyena', ''), ('hyena', '5'),
           ('g-both', ''), ('g-both', '5')]
SEEDS = [0, 1]
for seed in SEEDS:
    for m, attn in CONFIGS:
        train(m, seed, attn=attn)

## Results

'vs attention' is per seed; positive means worse than the all-attention model.

In [ ]:
from grassman import analyze
rows = analyze.load(RUNS)
print(analyze.val_table(rows, 'attention'))

mean = lambda n: sum(r['best_val_ppl'] for r in rows if analyze.name(r) == n) / max(1, sum(analyze.name(r) == n for r in rows))
a = mean('attention')
print('\nshare of the gap to attention recovered by one attention layer:')
for m in ['grassmann', 'concat32', 'hyena', 'g-both']:
    local, hyb = mean(m), mean(m + '+attn@5')
    print(f"  {m:<10} local {local:6.1f} -> hybrid {hyb:6.1f}   recovered {100 * (local - hyb) / (local - a):4.0f}%")
print(f"  concat32 with attention at layer 3 instead of 6: {mean('concat32+attn@2'):.1f}")

In [ ]:
import matplotlib.pyplot as plt
from grassman import analyze
rows = analyze.load(RUNS)
fig, ax = plt.subplots(figsize=(8, 4.5))
for r in rows:
    h = r['history']
    ax.plot([e['epoch'] for e in h], [e['val_ppl'] for e in h], ls='-' if r['seed'] == 0 else '--',
            label=f"{analyze.name(r)} s{r['seed']}")
ax.set_xlabel('epoch'); ax.set_ylabel('validation perplexity'); ax.grid(alpha=.3); ax.legend(fontsize=7, ncol=2)
plt.savefig(RUNS.replace('.jsonl', '_curves.png'), dpi=150, bbox_inches='tight'); plt.show()

## How to read it

- **Recovered ≥ 80%:** one attention layer does most of the work, and the local mixer is a cheap supporting part, as in StripedHyena and Jamba.
- **Local mixer type inside a hybrid:** if `grassmann`, `concat32`, `hyena` and `g-both` hybrids are all close, use the cheapest one. That is the answer for the Hyena-audio striped design too.
- **Layer 3 vs layer 6:** whether the attention layer's position matters.
- 5 epochs, 2 seeds: a pilot, not a final result.

Download the results before the runtime is deleted.

In [ ]:
!cd /content && zip -qr results.zip results
from google.colab import files
files.download('/content/results.zip')